
# <center> **Global Markets,Investments & Analytics**


## <center> BUS 520 | MBA 
### <center> Becamex Business School

### <center> Kang Seung Won (kwon.kang@eiu.edu.vn)
##### <center> Miss Do Thanh Thuy Vy (vy.do@eiu.edu.vn), Mr. Lam Truong Son (son.lam@eiu.edu.vn)

---
## <center> **Week 2 (Course Week 8): Performance Analysis & Customer Analytics**

---
## Learning Objectives: Week 2 (Course Week 8)

**Main Question:** How can managers use real-world data to make better decisions?

1. **Review:** Review key concepts from the previous week.

2. **Regression Analysis:** Estimate and interpret relationships between variables using Ordinary Least Squares (OLS) and Logistic Regression (Logit).

3. **Financial Performance and Risk:** Calculate and interpret daily returns, volatility, the Sharpe ratio, and beta, and apply the Capital Asset Pricing Model (CAPM).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
sns.set_style("whitegrid")

mpl.rcParams['figure.figsize'] = (12, 6)
mpl.rcParams['axes.grid'] = True

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from faker import Faker

np.random.seed(42)
N = 3000

s1 = np.clip(np.random.normal(8.7, 0.9, N), 4, 10)
s2 = np.clip(np.random.normal(8.6, 0.9, N), 4, 10)
s3 = np.clip(np.random.normal(8.7, 0.9, N), 4, 10)
total = s1 + s2 + s3

total = total * (28.0 / total.mean())
ratio = total / (s1+s2+s3)
s1, s2, s3 = s1*ratio, s2*ratio, s3*ratio

base = 1.0 + 3.0 * np.random.beta(6, 3, N)  # mean 3.0 exactly
gpa = base + 0.05 * (total - 28) + np.random.normal(0, 0.15, N)
gpa = np.clip(gpa, 1.0, 4.0).round(2)

s1 = np.clip(s1, 0, 10).round(1)
s2 = np.clip(s2, 0, 10).round(1)
s3 = np.clip(s3, 0, 10).round(1)
total = (s1+s2+s3).round(1)

EIU_population = pd.DataFrame({
    "student_id": [f"{24}{i+1:04d}" for i in range(N)],
    "exam_s1": s1,
    "exam_s2": s2,
    "exam_s3": s3,
    "exam_total": total,
    "gpa": gpa
})
EIU_population

---
# From Sample to Population: Regression and Classification


## The goal is to understand **the functional relationship**:

| Method | X (inputs) | Y (outcome) | Question |
|---|---|---|---|
| **Linear regression (OLS)** | `exam_total` | `gpa` | How is entrance-exam score related to GPA? |
| **Logistic regression (Logit)** | `gpa`, `exam_total` | `ielts_pass` = 0 or 1 | Do these scores help explain whether a student gets an IELTS score **above 6.0**? |



In [ ]:
# Keep the original 3,000 students. Add invented IELTS data.
from datetime import date

fake = Faker("vi_VN")
Faker.seed(2026)
rng_ielts = np.random.default_rng(2026)

EIU_population = EIU_population.copy()
student_count = len(EIU_population)

EIU_population["student_name"] = [fake.name() for _ in range(student_count)]
EIU_population["ielts_exam_date"] = [
    fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 10, 31))
    for _ in range(student_count)
]
EIU_population["ielts_study_months"] = rng_ielts.integers(4, 21, student_count)

# The score is invented for teaching, with random variation.
raw_score = (5.90
             + 0.55 * (EIU_population["gpa"] - 3.0)
             + 0.16 * (EIU_population["exam_total"] - 28.0)
             + rng_ielts.normal(0, 0.65, student_count))

# IELTS is reported in half-band steps; keep scores between 0 and 9.
EIU_population["ielts_score"] = np.clip(np.round(raw_score * 2) / 2, 0, 9)
EIU_population["ielts_pass"] = (EIU_population["ielts_score"] > 6.0).astype(int)

# Extra label for our earlier one-year discussion (NOT used as Logit Y).
EIU_population["ielts_pass_within_1year"] = (
    (EIU_population["ielts_score"] > 6.0)
    & (EIU_population["ielts_study_months"] <= 12)
).astype(int)

print("Invented population size:", len(EIU_population))
print("IELTS pass rate (score > 6):", f'{EIU_population["ielts_pass"].mean():.1%}')
display(EIU_population[["student_name", "exam_total", "gpa",
                        "ielts_score", "ielts_pass",
                        "ielts_study_months", "ielts_pass_within_1year"]].head())


In [ ]:
student_sample = EIU_population.sample(n=300, random_state=2026).copy()

print("Population size:", len(EIU_population))
print("Population mean:", EIU_population['gpa'].mean().round(3))
print("Sample size:", len(student_sample))
print("Sample GPA mean:", round(student_sample["gpa"].mean(), 3))


In [ ]:
student_sample.to_csv("student_sample.csv")


| Method | X (inputs) | Y (outcome) | Question |
|---|---|---|---|
| **Linear regression (OLS)** | `exam_total` | `gpa` | How is entrance-exam score related to GPA? |**

##### Calculation of OLS Coefficients
### $$ y = X\beta + \epsilon $$

- **Mean Squared Error (MSE)**: $ MSE(\beta) = \frac{1}{n}\sum_{i=1}^{n} (y_i - x_i\beta)^2 = \frac{1}{n}\epsilon'\epsilon = \frac{1}{n}(y - X\beta)'(y - X\beta) $
- Expand MSE: $ MSE(\beta) = \frac{1}{n}\left(y'y - \beta'X'y - y'X\beta + \beta'X'X\beta\right) $
- (Since $y'X\beta$ is scalar, $y'X\beta = \beta'X'y$)
- Simplify, $ MSE(\beta) = \frac{1}{n}\left(y'y - 2\beta'X'y + \beta'X'X\beta\right) $
- Differentiate w.r.t. $\beta$, $ \frac{\partial MSE(\beta)}{\partial \beta} = \frac{1}{n}\left(-2X'y + 2X'X\beta\right) $
- Set Derivative to Zero (Normal Equations)
$ \frac{1}{n}\left(-2X'y + 2X'X\beta\right) = 0 \implies -2X'y + 2X'X\beta = 0 \implies X'X\beta = X'y $
- Solve for $\beta$, 

$$
\hat{\beta}_{OLS} = \underset{\beta}{\text{argmin}} \frac{1}{n}\sum_{i=1}^{n} (y_i - x_i\beta)^2 = (X'X)^{-1}X'y
$$

In [ ]:
import statsmodels.api as sm

In [ ]:
x_ols = student_sample['exam_total']
x_ols = sm.add_constant(student_sample['exam_total'])
x_ols

In [ ]:
y_ols = student_sample['gpa']


$$
\hat{\beta}_{OLS} = (X'X)^{-1}X'y
$$

In [ ]:
ols = sm.OLS(y_ols, x_ols).fit()
print(ols.summary())

In [ ]:
X_pop = sm.add_constant(EIU_population[["exam_total"]])
y_pop = EIU_population["gpa"]

ols_pop = sm.OLS(y_pop, X_pop).fit()
print(ols_pop.summary())

population_beta = ols_pop.params["exam_total"]
population_beta

In [ ]:
N = 300
tries = 1000
betas = []

for i in range(tries):

    student_sample_sim = EIU_population.sample(n=N, replace=True)
    X_ols_sim = sm.add_constant(student_sample_sim[["exam_total"]])
    y_ols_sim = student_sample_sim["gpa"]

    ols_sim = sm.OLS(y_ols_sim, X_ols_sim).fit()
    betas.append(ols_sim.params["exam_total"])

betas = np.array(betas)
print('beta mean:', betas.mean())

In [ ]:
betas.round(3)

In [ ]:
sns.histplot(betas, bins=30, kde=True, stat="density")

plt.axvline(betas.mean(), color="blue",
            linestyle="--", label="Mean of Estimated Betas")
plt.axvline(population_beta, color="red",
            linestyle="--", label="Popluation Beta (True)")

plt.title(f"Sampling Distribution of OLS Beta (N={N})")
plt.xlabel("Estimated Beta")
plt.legend()
plt.show()

print('mean of beta from samples (1000 times):', betas.mean())
print('population beta (true value):', population_beta)

## From a Sample to a POPULATION

This is the **same idea** as sampling means. If we took another random sample of 240 students, our fitted slope would change.

- **Sample slope $b_1$:** our best estimate from these students.
- **Population slope $\beta_1$:** the relationship we want to learn about for all 3,000 students.
- **SE of the slope:** how much the slope estimate would vary across samples (approximately).
- **95% confidence interval:** a range of possible population slopes compatible with our data and model.
- **p-value:** how strongly the sample contradicts a proposed **zero population slope**.

This **SE** is *not* the earlier `SD / sqrt(n)` for a sample mean. The regression program calculates the **coefficient's own SE**, considering the scatter around the line and spread of X values.


### Calculate the standard error of the OLS slope

Our fitted line for the **240 students** is $\widehat{GPA}_i=b_0+b_1\,exam\_total_i$. The **population slope** $\beta_1$ is fixed but unknown; the **sample slope** $b_1$ (also written $\hat\beta_1$) changes across random samples. We estimate the standard deviation of that sampling distribution with $SE(b_1)$.

For **simple OLS with an intercept and one predictor** (`exam_total`), the usual textbook SE is:

$$\boxed{SE(b_1)=\sqrt{\frac{s^2}{S_{xx}}}=\frac{s}{\sqrt{\sum_{i=1}^{n}(x_i-\bar x)^2}}},\qquad S_{xx}=\sum_{i=1}^{n}(x_i-\bar x)^2.$$

Follow the five steps below **using our actual sample**:

1. **Predict GPA for each student.** $\hat y_i=b_0+b_1x_i$.
2. **Find each residual.** $e_i=y_i-\hat y_i$ is the vertical distance from a student's observed GPA to the fitted line.
3. **Measure unexplained GPA variation.** Add up squared residuals, $SSE=\sum e_i^2$, then estimate error variance $s^2=\dfrac{SSE}{n-2}$. We divide by **$n-2$** because we estimated **two coefficients** (intercept and slope). Its square root, $s$, is the *residual standard error* in GPA units.
4. **Measure the spread of exam scores.** $S_{xx}=\sum(x_i-\bar x)^2$; a wider range of X values makes the slope easier to estimate precisely.
5. **Combine the two parts.** $SE(b_1)=\sqrt{\dfrac{SSE/(n-2)}{S_{xx}}}$. Larger residual scatter **increases** SE; wider X spread **decreases** SE, all else equal.

The regression package computes this for us; next we reproduce it **manually** and check that it matches `ols.bse["exam_total"]`. Note that this is **not** the sample-mean formula `SD / sqrt(n)`.


## p-value for a regression slope

Ask a specific question about the **population**:

$$H_0: \beta_1=0\qquad H_1: \beta_1\ne0$$

- **$H_0$:** in the population, there is no *linear* association between exam total and GPA (zero slope).
- **$H_1$:** the population slope is not zero.


In [ ]:
print(ols.params['exam_total'])
print(ols.bse['exam_total'])

In [ ]:
t_score = (ols.params['exam_total'] - 0) / ols.bse['exam_total']
t_score

In [ ]:
# p-value
from scipy import stats

p_value = 2 * stats.t.sf(abs(t_score), df=len(x_ols)-2)
p_value

In [ ]:
ols.tvalues['exam_total']
ols.pvalues['exam_total']
ols.conf_int(alpha=0.05).loc['exam_total']

In [ ]:
print(ols.summary())

---
## Linear Regression with Real Data: House Prices 
### House prices (`hprice1`)

**Business question:** How much is an additional square foot of house space associated with the selling price, after accounting for lot size and bedrooms?

88 houses sold in the Boston area in 1990.

| Variable | Meaning |
|---|---|
| `price` (Y) | Selling price (thousand US dollars) |
| `sqrft` (X1) | House size (square feet) |
| `lotsize` (X2) | Lot size (square feet) |
| `bdrms` (X3) | Number of bedrooms |

[Download CSV](https://vincentarelbundock.github.io/Rdatasets/csv/wooldridge/hprice1.csv) · [Variable descriptions](https://justinmshea.github.io/wooldridge/reference/hprice1.html)

**Data access:** The following code downloads CSV files directly. If classroom Wi-Fi is unavailable, download them beforehand and place `hprice1.csv`, `ceosal1.csv` and `coke.csv` in the notebook's working directory. No special dataset package is needed.



In [ ]:
hprice1 = pd.read_csv("https://vincentarelbundock.github.io/Rdatasets/csv/wooldridge/hprice1.csv")
hprice1

In [ ]:
hprice1.isna().sum()

### Simple regression: price and house size

$$price_i=\beta_0+\beta_1 sqrft_i+u_i$$


In [ ]:
house_data = hprice1[["price", "sqrft", "bdrms"]].dropna()
house_simple = sm.OLS(house_data["price"], sm.add_constant(house_data[["sqrft"]])).fit()
print(house_simple.summary())

In [ ]:
# Feature Selection / Feature Extraction
# house_feature = sm.OLS(house_data["price"], sm.add_constant(house_data[["sqrft", 'bdrms']])).fit()
house_feature = sm.OLS(house_data["price"], sm.add_constant(house_data[['bdrms']])).fit()
print(house_feature.summary())

In [ ]:
house_data.corr()

---
## **Financial Performance and Risk**
### Business case: Should an investor choose VinFast or a US market fund?

- **VinFast (VFS):** A Vietnamese electric vehicle manufacturer listed on Nasdaq on August 15, 2023.
- **SPY:** An exchange-traded fund (ETF) that tracks the S&P 500 Index.

**Source pages:**
- Yahoo Finance, VFS: https://finance.yahoo.com/quote/VFS/history/
- Yahoo Finance, SPY: https://finance.yahoo.com/quote/SPY/history/
- FRED, 3-month Treasury yield (`DGS3MO`): https://fred.stlouisfed.org/series/DGS3MO


In [ ]:
import yfinance as yf

tickers = ['VFS', 'SPY']
prices = yf.download(tickers, start="2023-08-15", auto_adjust=True, progress=False)['Close'].dropna()
prices

In [ ]:
prices.plot();

In [ ]:
prices.apply(lambda x: (x - x.mean()) / x.std()).plot();
prices.last('2Y').apply(lambda x: (x - x.mean()) / x.std()).plot();
prices.last('1Y').apply(lambda x: (x - x.mean()) / x.std()).plot();

## Understanding Data: 

In [ ]:
prices.describe()

---
### 1) Stock return

**Daily return:** How much did the price change since the previous trading day?

$$r_t = (P_t - P_{t-1}) / P_{t-1} = P_t/P_{t-1}-1$$

**Total return over our sample:** Compare the last adjusted price with the first price.

$$R = P_{end}/P_{start}-1$$

In [ ]:
d_return = (prices / prices.shift(1)) - 1
d_return

In [ ]:
prices.pct_change()

In [ ]:
returns = prices.pct_change().dropna()
total_return = (prices.iloc[-1] / prices.iloc[0]) - 1
print("Total return over the selected period:", (total_return * 100).round(2))

In [ ]:
returns.describe()

In [ ]:
returns['VFS'].idxmin()

---
## Volatility

- **Volatility:** How much daily returns change around their average.
- **Annualized volatility:** Daily standard deviation × $\sqrt{252}$.

In [ ]:
returns.plot();

fig, axes = plt.subplots(2, 1)

returns['VFS'].plot(ax=axes[0], title='Vinfast')
returns['SPY'].plot(ax=axes[1], title='SPY 500')
plt.show();


In [ ]:
annual_volatility = returns.std() * np.sqrt(252)

risk_table = pd.DataFrame({
    "Annual volatility (%)": annual_volatility * 100,
})
print(risk_table.round(2))
print('risk comparison:', risk_table.iloc[1]/risk_table.iloc[0])


---
## Sharpe Ratio: return above a low-risk rate

The Sharpe Ratio measures reward per unit of risk: how much extra return (above a safe, low-risk investment) you earned for every unit of volatility you endured. A higher Sharpe ratio means better risk-adjusted performance.

**Question:** How much extra return did the investor get for the return volatility?

- For this class, we use the **US 3-month Treasury constant-maturity yield** as a *proxy* for the risk-free rate.
- FRED series: **DGS(daily government security)3MO**. It is an **annual percentage yield**, not a daily return.

| Treasury Maturity | Common Use |
|---|---|
| 3-month (`DGS3MO`) | Risk-free rate for Sharpe ratio and short-term investment analysis |
| 2-year (`DGS2`) | Monetary policy expectations and interest-rate analysis |
| 5-year (`DGS5`) | Medium-term interest rates and bond market analysis |
| 10-year (`DGS10`) | Long-term interest rates, macroeconomic analysis, and valuation |
| 30-year (`DGS30`) | Long-term financing, pensions, and duration analysis |

**Steps:**
1. Read historical Treasury yields from FRED.
2. Convert each annual yield to a one-day rate.
3. Subtract the daily low-risk rate from each daily stock return.
4. Divide average daily excess return by daily volatility; multiply by $\sqrt{252}$.

$$Sharpe = \frac{\overline{r-r_f}}{s(r)}\sqrt{252}$$


In [ ]:
# FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO"
# FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS5"
fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO")
fred

In [ ]:
fred.isnull().sum()

In [ ]:
fred.info()
fred.dtypes
fred.describe()

In [ ]:
fred.columns = ["Date", "AnnualYieldPercent"]
fred

In [ ]:
fred["Date"] = pd.to_datetime(fred["Date"]) 
fred["AnnualYieldPercent"] = fred["AnnualYieldPercent"].ffill()
fred

In [ ]:
# Match with stock return dates
rf = fred.set_index("Date")["AnnualYieldPercent"]
rf = rf.reindex(returns.index, method="ffill")
# rf.isna().sum()

# Convert annual percentage yield to daily rate
rf_daily = rf / 100 / 252

In [ ]:
rf_daily

In [ ]:
data = returns.copy()
data["RF_Daily"] = rf_daily

data.head(10)


$$Sharpe = \frac{\overline{r-r_f}}{s(r)}\sqrt{252}$$

In [ ]:
stock_returns = data[["SPY", "VFS"]]

excess_daily = stock_returns.sub(data["RF_Daily"], axis=0) # axis=0: row-wise, axis=1: column wise
excess_daily 

sharpe_ratio = ((excess_daily.mean()/ stock_returns.std())* np.sqrt(252))
sharpe_ratio

---
## CAPM (Captital Asset Pricing Model) and beta


CAPM estimates the expected return of an investment based on its systematic (market) risk.

$$
\boxed{E(R_i)=R_f+\beta_i[E(R_m)-R_f]}
$$

where:

- $E(R_i)$: Expected return of asset $i$
- $R_f$: Risk-free rate
- $\beta_i$: Sensitivity of asset $i$ to market movements. Beta measures systematic (market) risk—how sensitive a stock (VFS) is to overall market movements (SPY), normalized by the market's own variance. 
- $E(R_m)$: Expected market return
- $E(R_m)-R_f$: Market risk premium

**Main Question:** What return should an investor expect as compensation for taking systematic risk?

- **VFS:** Individual asset under consideration.
- **SPY:** Proxy for the market portfolio.
- **US Treasury Bill Rate:** Proxy for the risk-free rate.

In [ ]:
(returns * 100).round(2)

In [ ]:
returns.plot();

### Estimate beta with OLS regression


$$R_{VFS,t}=a+\beta R_{SPY,t}+e_t$$

- **a (intercept):** Fitted VFS return when SPY return is zero.
- **Beta (slope):** Change in the fitted VFS return for a one-unit change in SPY return. Beta measures systematic (market) risk—how sensitive a stock (VFS) is to overall market movements (SPY), normalized by the market's own variance. 
- **e (error):** Part of VFS's return not explained by this fitted line.

$$\beta=\frac{Cov(R_{VFS},R_{SPY})}{Var(R_{SPY})}$$



In [ ]:
beta_from_covariance = returns["VFS"].cov(returns["SPY"]) / returns["SPY"].var()
beta_from_covariance

In [ ]:

X_beta = sm.add_constant(returns["SPY"])  
y_beta = returns["VFS"]
beta_model = sm.OLS(y_beta, X_beta).fit()

print(beta_model.summary())

In [ ]:
# X_beta = sm.add_constant(returns["SPY"])  
# y_beta = returns["VFS"]
beta = np.linalg.inv(X_beta.T@ X_beta )@ X_beta.T @ y_beta
beta

In [ ]:
sns.regplot(x='SPY', y='VFS', data=returns);

### For $E(R_m)-R_f$


$$
\boxed{E(R_i)=R_f+\beta_i[E(R_m)-R_f]}
$$


- $E(R_i)$: Expected return of asset $i$
- $R_f$: Risk-free rate --> **US Treasury Bill Rate:** Proxy for the risk-free rate.
- $\beta_i$: Sensitivity of asset $i$ to market movements. Beta measures systematic (market) risk—how sensitive a stock (VFS) is to overall market movements (SPY), normalized by the market's own variance. 
- $E(R_m)$: Expected market return --> **SPY return**
- $E(R_m)-R_f$: Market risk premium

In [ ]:
returns

In [ ]:
returns = prices.pct_change().dropna()
total_return = prices.iloc[-1] / prices.iloc[0] - 1
total_return[0]

### Compound Annual Growth Rate (CAGR)


Converts a multi-year cumulative return (`total_return`) into a constant annual rate of return ($R_m$) accounting for compounding:

$$\text{CAGR} = (1 + \text{total\_return})^{\frac{1}{\text{years}}} - 1$$

- **$1 + \text{total\_return}$:** The total wealth multiplier over the entire period (e.g., $1 + 0.6888 = 1.6888$).
- **$^{\frac{1}{\text{years}}}$:** Takes the $n$-th root to find the per-year compounding growth factor.
- **$- 1$:** Subtracts the initial principal to express the result as a percentage return.

#### Simple Example
Suppose you invest **$100** in a stock, and after **2 years**, it grows to **$144**:

1. **Find the Total Return:**
   $$\text{total\_return} = \frac{\$144 - \$100}{\$100} = 0.44 \quad (44\%)$$
2. **Add 1 (Wealth Multiplier):**
   $$1 + 0.44 = 1.44$$
3. **Take the $n$-th Root ($\frac{1}{\text{years}} = \frac{1}{2}$):**
   $$(1.44)^{\frac{1}{2}} = \sqrt{1.44} = 1.20$$
4. **Subtract 1:**
   $$\text{CAGR} = 1.20 - 1 = 0.20 \quad (\mathbf{20\% \text{ per year}})$$

**Verify Compounding:**
- **Year 1:** $\$100 \times 1.20 = \$120$
- **Year 2:** $\$120 \times 1.20 = \$144$



In [ ]:
# Calendar days divided by 365.25 (accounts for leap years)
years = (prices.index[-1] - prices.index[0]).days / 365.25
years

In [ ]:

Rm = (1 + total_return[0]) ** (1/years) - 1
print(f"Annualized RM: {Rm:.2%}")

In [ ]:
total_return

In [ ]:
rf

In [ ]:
rf_period = rf.loc[prices.index[0]:prices.index[-1]]

rf_period

In [ ]:
# Market proxy: SPY
# Risk-free proxy: 3-month Treasury bill

# Annualized historical SPY return
years = (prices.index[-1] - prices.index[0]).days / 365.25
Rm = (1 + total_return["SPY"]) ** (1 / years) - 1

# Average risk-free rate over the same period
rf_period = rf.loc[prices.index[0]:prices.index[-1]]
rf_avg_annual = rf_period.mean() / 100
rf_latest_annual = rf.iloc[-1] / 100

# Calculate historical market risk premium
market_risk_premium = Rm - rf_avg_annual

# CAPM returns using historical premium
beta_vfs = beta_model.params["SPY"]
capm_required_vfs = rf_latest_annual + beta_vfs * market_risk_premium
capm_required_spy = rf_latest_annual + market_risk_premium

# results
print("Annualized SPY return:", Rm.round(2))
print("Average historical risk-free rate:", rf_avg_annual.round(2))
print("Historical market risk premium:", market_risk_premium.round(2))
print("Latest risk-free rate:", rf_latest_annual.round(2))
print("Historical VFS beta:", beta_vfs.round(2))
print("CAPM required VFS return:", capm_required_vfs.round(2))
print("CAPM required SPY return:", capm_required_spy.round(2))

---
# Group Assignment #3: CAPM in Vietnam

---
# Logistic Regression (Logit): GPA + Exam Total + Study Month → IELTS Pass 

## Why is this classification and not linear regression?

Here the outcome is not a GPA number. It is **yes/no**:

- **Y = `ielts_pass` = 1** if invented IELTS score **> 6.0**.
- **Y = `ielts_pass` = 0** if invented IELTS score **≤ 6.0**.

**Logistic regression (Logit)** predicts a probability between **0% and 100%**. It is a regression *model* used for **binary classification**.


### Why take the log of odds?

Linear regression predicts a number using a straight line. But when Y is 0 or 1, a straight line can predict below 0 or above 1. We need a way to convert **any real-valued score** into a valid probability.


In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=100, n_features=2, n_classes=2, n_redundant=0, random_state=123)

plt.scatter(X[:, 0], y);

In [ ]:
## Linear Fitting

sns.regplot(x=X[:, 0], y=y);
plt.title("Linear Fitting or Linear Probability Model");
plt.ylabel('Probability')

#### Linear Probability Model (LPM)
- **Advantages:**
    - Simple and Easy to Interpret
- **Disadvantages:**
    - Can yield predicted probabilities outside the [0, 1] range, making them invalid.
    - Suffers from heteroscedasticity, meaning the variance of the error term is not constant across all values of the independent variables, which can lead to inefficient estimates and incorrect standard errors.


---
## Solution

Let $p=P(Y=1\mid X)$, the probability of an event (e.g., IELTS pass).

$$\text{Odds}=\frac{p}{1-p},\qquad \text{Logit}=\log\left(\frac{p}{1-p}\right)$$


**2. Linear combination (the same starting idea as OLS)**

$$z=w_0+w_1x_1+\cdots+w_kx_k=\mathbf{w}^{T}\mathbf{X}$$

Here the vector $\mathbf{X}=(1,x_1,\ldots,x_k)^T$ includes a 1 for the intercept. Logistic regression sets

$$\log\left(\frac{p}{1-p}\right)=z.$$

When $x_j$ increases by one unit, log-odds change by $w_j$, **holding all other predictors fixed**. Odds are multiplied by $e^{w_j}$. Probability does **not** change by a fixed amount.

**3. Solve for probability: the sigmoid function**

Take the exponential of both sides and rearrange:

$$\frac{p}{1-p}=e^z$$
$$p=e^z(1-p)$$
$$p(1+e^z)=e^z$$
$$\boxed{p=\frac{e^z}{1+e^z}=\frac{1}{1+e^{-z}}=\sigma(z)}$$

This S-shaped **sigmoid** converts any real score $z$ into a probability between 0 and 1. When $z=0$, $p=0.5$; negative scores give probabilities below 0.5 and positive scores above 0.5.

In [ ]:
p = np.arange(0, 1, 0.01)
odds = p/(1-p)
np.log(odds).round(2)

In [ ]:
plt.plot(np.log(odds), p);
plt.axhline(y=0.5, color='r', linestyle='--');

In [ ]:
X = student_sample[['exam_total', 'gpa', 'ielts_study_months']]
y = student_sample['ielts_pass']

X = sm.add_constant(X)
ielts_logit = sm.Logit(y, X).fit()

print(ielts_logit.summary())

In [ ]:
import statsmodels.formula.api as smf

ielts_logit = smf.logit(formula="ielts_pass ~ exam_total + gpa + ielts_study_months", data=student_sample).fit()

print(ielts_logit.summary())

In [ ]:
# Interpretation of coefficients
ielts_logit.params, np.exp(ielts_logit.params), (np.exp(ielts_logit.params)-1)*100

## exp(β)?

$$
\log(Odds)=\beta_0+\beta_1X
$$

Before a one-unit increase in $X$:

$$
\log(Odds_{old})=\beta_0+\beta_1X
$$

After a one-unit increase in $X$:

$$
\log(Odds_{new})=\beta_0+\beta_1(X+1)
$$

Subtracting the two equations:

$$
\log(Odds_{new})-\log(Odds_{old})=\beta_1 : \log(A)-\log(B)=\log\left(\frac{A}{B}\right)
$$

$$
\log\left(\frac{Odds_{new}}{Odds_{old}}\right)=\beta_1
$$

Taking the exponential of both sides:

$$
\boxed{
\frac{Odds_{new}}{Odds_{old}}=e^{\beta_1}
}
$$


The standard formula for percentage change is:

$$
\text{Percentage Change}
= \frac{\text{New Value} - \text{Original Value}}
{\text{Original Value}} \times 100
$$

Rearranging the equation:

$$
= \left(
\frac{\text{New Value}}{\text{Original Value}} - 1
\right) \times 100
$$

Substituting the odds ratio:

$$
\boxed{
\text{Percentage Change in Odds} = (e^\beta - 1)\times 100
}
$$

The estimated coefficient of `exam_total` is:

$$
\beta = 0.283617
$$

The odds ratio is:

$$
e^{0.283617} = 1.327925
$$


$$
(e^\beta - 1)\times 100
= (1.327925 - 1)\times 100
= 32.79\%
$$

**Interpretation:** A one-point increase in the entrance exam
score is associated with a 32.79% increase in the odds of
passing IELTS, holding all other variables constant.



In [ ]:

pd.DataFrame({
    'Coefficient': ielts_logit.params,
    'Odds Ratio': np.exp(ielts_logit.params),
    'Change in Odds (%)': (np.exp(ielts_logit.params) - 1) * 100,
    'P-value': ielts_logit.pvalues
}).round(4)

---
# Logistic Regression with Real Data: Coke or Pepsi?


The `coke` dataset from *Principles of Econometrics* contains scanner data on 1,140 individuals who bought Coke or Pepsi. It is historical teaching data, not a current market survey.

| Variable | Meaning |
|---|---|
| `coke` (Y) | 1 = Coke chosen; 0 = Pepsi chosen |
| `pratio` (X1) | Price of Coke / price of Pepsi |
| `disp_pepsi` (X2) | 1 = Pepsi on display; 0 = not displayed |
| `disp_coke` (X3) | 1 = Coke on display; 0 = not displayed |


[Download CSV](https://www.principlesofeconometrics.com/poe5/data/csv/coke.csv) · [Variable descriptions](https://rdrr.io/github/ccolonescu/PoEdata/man/coke.html)

In [ ]:
coke = pd.read_csv("https://www.principlesofeconometrics.com/poe5/data/csv/coke.csv")
coke

In [ ]:
coke['coke'].value_counts()
coke['coke'].value_counts(normalize=True)

## Estimation


$$\log\left(\frac{p}{1-p}\right)=\beta_0+\beta_1 pratio+\beta_2 disp\_pepsi+\beta_3 disp\_coke$$

where $p=P(coke=1\mid X)$.

In [ ]:
coke_logit = sm.Logit(coke['coke'], sm.add_constant(coke[['pratio', 'disp_pepsi', 'disp_coke']])).fit()
print(coke_logit.summary())

In [ ]:
coke_logit.params, np.exp(coke_logit.params), (np.exp(coke_logit.params) - 1)*100

In [ ]:
coke['pr_demean'] = coke['pratio']-coke['pratio'].mean()
coke_logit_demean = sm.Logit(coke['coke'], sm.add_constant(coke[['pr_demean', 'disp_pepsi', 'disp_coke']])).fit()
print(coke_logit_demean.summary())

In [ ]:
coke_logit_demean.params, np.exp(coke_logit_demean.params), (np.exp(coke_logit_demean.params)-1)*100

In [ ]:
new_data = pd.DataFrame({
    'const': [1],
    'pr_demean': [0],
    'disp_pepsi': [0],
    'disp_coke': [1]
})

coke_logit_demean.predict(new_data)

In [ ]:
scenarios = pd.DataFrame({
    'const': [1, 1, 1, 1],
    'pr_demean': [0, 0, 0, 0],
    'disp_pepsi': [0, 1, 0, 1],
    'disp_coke': [0, 0, 1, 1]
})

scenarios['prob_coke'] = coke_logit_demean.predict(scenarios)
scenarios['prob_pepsi'] = 1 - scenarios['prob_coke']

scenarios.round(4)

In [ ]:
price_scenarios = pd.DataFrame({
    'const': 1,
    'pr_demean': np.linspace(-0.5, 0.5, 100),
    'disp_pepsi': 0,
    'disp_coke': 0
})

price_scenarios['prob_coke'] = (
    coke_logit_demean.predict(price_scenarios)
)

plt.plot(
    price_scenarios['pr_demean'],
    price_scenarios['prob_coke']
)

plt.xlabel('Relative Price (Demeaned)')
plt.ylabel('Probability of Purchasing Coke')
plt.title('Predicted Probability of Purchasing Coke')
plt.grid(alpha=0.3)
plt.show()


---
### Multiclass Logistic Regression: Multinomial Logit (Softmax Function)
- **$N$ classes**, where $N > 2$: 
- For each class $i$ (from $1$ to $N$), we compute a linear score $z_i$:
$$z_{i}=\mathbf{w}_{i}^{T}\mathbf{x}+b_{i}\quad \text{for\ }i=1,\dots ,N$$

- Define Log-Odds Ratios Relative to a Reference Class
$$\log \left(\frac{P(Y=i|\mathbf{x})}{P(Y=N|\mathbf{x})}\right)=z_{i}$$

- Exponentiate to Find Probability Ratios
$$\frac{P(Y=i|\mathbf{x})}{P(Y=N|\mathbf{x})}=e^{z_{i}}$$
$$\Rightarrow P(Y=i|\mathbf{x})=P(Y=N|\mathbf{x})\cdot e^{z_{i}}\quad \text{for\ all\ }i\in \{1,\dots ,N\}$$

- Use the Sum-to-One Constraint for Normalization
    - $\sum _{i=1}^{N}P(Y=i|\mathbf{x})=1$
    - $\rightarrow \sum _{i=1}^{N}\left(P(Y=N|\mathbf{x})\cdot e^{z_{i}}\right)=1$
    - $\rightarrow P(Y=N|\mathbf{x})$ from the summation: $P(Y=N|\mathbf{x})\sum _{i=1}^{N}e^{z_{i}}=1$

- Solve for the probability of the reference class $P(Y=N|\mathbf{x})$:
$$P(Y=N|\mathbf{x})=\frac{1}{\sum _{i=1}^{N}e^{z_{i}}}$$

- Substitute Back to Find the General Formula
    - Substitute the derived expression for $P(Y=N|\mathbf{x})$ back into the general probability ratio from Step 3 to find the formula for the probability of any class $i$: $$P(Y=i|\mathbf{x})=P(Y=N|\mathbf{x})\cdot e^{z_{i}}=\frac{e^{z_{i}}}{\left(\sum _{j=1}^{N}e^{z_{j}}\right)}$$   

    - Note: The index $j$ in the denominator is used to avoid confusion, as it includes every class, including $i$.

---
### Application: Multinomial Logit for Yogurt Brand Choice

**Main Business Question:**

How do prices and advertising influence consumers'
brand choice probabilities?

Using Multinomial Logistic Regression, management can
estimate consumer choice probabilities and simulate
alternative pricing and advertising strategies.

In [ ]:
# load dataset
yogurt = pd.read_csv("https://vincentarelbundock.github.io/Rdatasets/csv/Ecdat/Yogurt.csv")
yogurt

In [ ]:
yogurt['id'].nunique()

In [ ]:
yogurt.groupby('id').size()

In [ ]:
print(yogurt['choice'].value_counts())
print(yogurt['choice'].value_counts(normalize=True)*100)


**Dannon as the reference category**.

$$
\log\left(\frac{P(Yoplait)}{P(Dannon)}\right)=z_1, \text{ }

\log\left(\frac{P(Weight)}{P(Dannon)}\right)=z_2,\text{ }

\log\left(\frac{P(Hiland)}{P(Dannon)}\right)=z_3
$$

#### Then $z_j$, for each non-reference brand, the linear score is defined as:

$$
z_j=\beta_{j0}+\boldsymbol{\beta}_j^T\mathbf{x}
$$

The reference category's linear score is normalized to zero: $z_0=0$,

$$
e^{z_0}=e^0=1
$$

#### the Softmax Function for the probability of choosing each brand 

**Dannon (Reference Category):** $P(Dannon)=\frac{1}{1+e^{z_1}+e^{z_2}+e^{z_3}}$,

**Yoplait:**: $P(Yoplait)=\frac{e^{z_1}}{1+e^{z_1}+e^{z_2}+e^{z_3}}$,

**Weight Watchers:** $P(Weight)=\frac{e^{z_2}}{1+e^{z_1}+e^{z_2}+e^{z_3}}$,

**Hiland:** $P(Hiland)=\frac{e^{z_3}}{1+e^{z_1}+e^{z_2}+e^{z_3}}$.



#### Interpretation of Estimated Coefficients

Each estimated coefficient measures the change in the log probability
ratio of choosing a particular brand relative to Dannon, holding
all other explanatory variables constant.

For an estimated coefficient $\beta$:

- $\beta$: Change in the log probability ratio.
- $e^\beta$: Relative risk ratio (RRR).
- $(e^\beta-1)\times100$: Percentage change in the relative probability ratio.



In [ ]:
brand_mapping = {'dannon': 0, 'yoplait': 1, 'weight': 2, 'hiland': 3}

y = yogurt['choice'].map(brand_mapping)

X = yogurt[[
    'price.dannon',
    'price.yoplait',
    'price.weight',
    'price.hiland',
    'feat.dannon',
    'feat.yoplait',
    'feat.weight',
    'feat.hiland'
]]

X = sm.add_constant(X)

In [ ]:
yogurt_mnlogit = sm.MNLogit(y, X).fit()
print(yogurt_mnlogit.summary())

In [ ]:
yogurt_mnlogit.params, np.exp(yogurt_mnlogit.params), (np.exp(yogurt_mnlogit.params) - 1) * 100

In [ ]:
predicted_prob = yogurt_mnlogit.predict(X)

predicted_prob.columns = [
    'Dannon',
    'Yoplait',
    'Weight',
    'Hiland'
]

predicted_prob.head()

#### Scenario: Dannon reduces its price by 10%

In [ ]:
X_new = X.copy()

X_new['price.dannon'] *= 0.9

new_prob = yogurt_mnlogit.predict(X_new)

new_prob.columns = predicted_prob.columns

comparison = pd.DataFrame({
    'Baseline (%)': predicted_prob.mean() * 100,
    'Price Reduction (%)': new_prob.mean() * 100,
    'Change': (
        new_prob.mean() - predicted_prob.mean()
    ) * 100
})

comparison.round(2)